# Mission 3 환자 증상 인식 — KLUE-RoBERTa 학습

**모델**: `klue/roberta-base` + mean pooling + 2층 분류기 (9개 증상 멀티라벨)

| 항목 | 설정 |
|---|---|
| 입력 | 대화 전사 텍스트(`text`)만 시간순으로 이어붙임 (화자 태그 미사용) |
| 출력 | 9개 증상 (고열, 구토, 두통, 복통, 어지러움, 열상, 오심, 전신쇠약, 호흡곤란) |
| Loss | BCEWithLogitsLoss + pos_weight (Train 음성/양성 비율로 계산) |
| 최적화 | AdamW (backbone 2e-5, head 1e-3), linear warmup 10%, AMP |
| 판정 | threshold 0.5 고정 |
| seed | 42, 123, 2026 |

- 학습은 Training(서울) 29,200건만 사용, Validation은 early stopping·best checkpoint 선정·앙상블 조합 선정에만 사용
- 최종 제출: 이 노트북의 KLUE-RoBERTa 3 seed + KcELECTRA 3 seed 확률 평균 앙상블 (Validation Macro F1 0.6543)
- 환경: Google Colab, Tesla T4

## 0. 데이터 준비

Mission 3는 텍스트만 사용하므로 zip에서 라벨링 JSON만 추출합니다.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, zipfile

ZIP_DIR  = '/content/drive/MyDrive/DCC_원본_zip'
WORK_DIR = '/content/drive/MyDrive/DCC/mission3/klue_roberta'   # 새 폴더 (여기에만 저장)
LOCAL_DIR = '/content/data'                                       # Colab 로컬 (세션 끝나면 사라짐)

# 덮어쓰기 방지: 폴더가 이미 있고 비어있지 않으면 경고
if os.path.exists(WORK_DIR) and os.listdir(WORK_DIR):
    print(f"⚠️ {WORK_DIR}에 이미 파일이 있습니다:", os.listdir(WORK_DIR)[:10])
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(LOCAL_DIR, exist_ok=True)

zips = sorted(glob.glob(f'{ZIP_DIR}/*.zip'))
print(f"zip {len(zips)}개")

n = 0
for zp in zips:
    with zipfile.ZipFile(zp) as z:
        for m in z.namelist():
            if m.endswith('.json'):
                z.extract(m, LOCAL_DIR)
                n += 1
print(f"JSON {n}개 추출 완료")

Mounted at /content/drive
⚠️ /content/drive/MyDrive/DCC/mission3/klue_roberta에 이미 파일이 있습니다: ['parsed_train.pkl', 'parsed_val.pkl', 'klue-roberta_meanpool_posw_seed42', 'klue-roberta_meanpool_none_seed42', 'klue-roberta_meanpool_sqrt_seed42', 'klue-roberta_attnpool_sqrt_seed42', 'klue-roberta_meanpool_sqrt_seed123', 'klue-roberta_meanpool_posw_seed123', 'klue-roberta_meanpool_posw_seed2026', 'klue-roberta_meanpool_none_seed2026']
zip 11개
JSON 32840개 추출 완료


In [ ]:
# 추출된 JSON 경로 목록 (Training / Validation 개수 확인)
all_json = sorted(glob.glob(f'{LOCAL_DIR}/**/*.json', recursive=True))
print(len(all_json))
for p in all_json[:3] + all_json[-3:]:
    print(p)

from collections import Counter
print(Counter(os.path.dirname(p).replace(LOCAL_DIR, '') for p in all_json).most_common(10))

32840
/content/data/대학부 데이터/Training/2.라벨링데이터/TL_서울_구급/651e464d69a4f266f062680b_20220101.json
/content/data/대학부 데이터/Training/2.라벨링데이터/TL_서울_구급/651e464d69a4f266f0626810_20220101.json
/content/data/대학부 데이터/Training/2.라벨링데이터/TL_서울_구급/651e464d69a4f266f062681a_20220101.json
/content/data/대학부 데이터/Validation/2.라벨링데이터/VL_서울_구급/651e5dd6049c168a19187bf5_20221105.json
/content/data/대학부 데이터/Validation/2.라벨링데이터/VL_서울_구급/6551f413f908f085d235557b_20220119.json
/content/data/대학부 데이터/Validation/2.라벨링데이터/VL_서울_구급/6551f6854f4d810f409c69e6_20220623.json
[('/대학부 데이터/Training/2.라벨링데이터/TL_서울_구급', 29200), ('/대학부 데이터/Validation/2.라벨링데이터/VL_서울_구급', 3640)]


In [ ]:
!pip install -q transformers

import json, random
import numpy as np
import pandas as pd
import torch

TARGET_SYMPTOMS = ["고열", "구토", "두통", "복통", "어지러움",
                   "열상", "오심", "전신쇠약", "호흡곤란"]   # 공식 9개, 순서 고정
MODEL_NAME = 'klue/roberta-base'
THRESHOLD = 0.5

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

# Training / Validation 파일 분리
train_files = [p for p in all_json if 'Training' in p]
val_files   = [p for p in all_json if 'Validation' in p]
print(len(train_files), len(val_files))   # 29200, 3640 이어야 함

29200 3640


## 1. 텍스트·라벨 파싱

In [ ]:
def build_input_text(data):
    """발화를 시작 시각 순으로 정렬해 text만 이어붙임"""
    utts = sorted(data['utterances'], key=lambda u: u['startAt'])
    return " ".join(u['text'] for u in utts)          # 화자 태그 없음 (규정)

def to_multihot(symptoms):
    """symptom 리스트 → 9차원 0/1 벡터 (9개 외 증상은 해당 라벨만 제외)"""
    return [1 if s in symptoms else 0 for s in TARGET_SYMPTOMS]

def parse_files(files):
    rows = []
    for fp in files:
        with open(fp, encoding='utf-8') as f:
            data = json.load(f)
        rows.append({
            'call_id': os.path.splitext(os.path.basename(fp))[0],
            'text': build_input_text(data),
            'labels': to_multihot(data.get('symptom', [])),
        })
    return pd.DataFrame(rows)

# 파싱 결과를 캐시해 재실행 시간 단축
CACHE_TRAIN = f'{WORK_DIR}/parsed_train.pkl'
CACHE_VAL   = f'{WORK_DIR}/parsed_val.pkl'

if os.path.exists(CACHE_TRAIN):
    train_df, val_df = pd.read_pickle(CACHE_TRAIN), pd.read_pickle(CACHE_VAL)
    print("캐시 로드")
else:
    train_df, val_df = parse_files(train_files), parse_files(val_files)
    train_df.to_pickle(CACHE_TRAIN); val_df.to_pickle(CACHE_VAL)
    print("파싱 후 캐시 저장")

print(len(train_df), len(val_df))
print(dict(zip(TARGET_SYMPTOMS, np.array(train_df['labels'].tolist()).sum(0))))
assert train_df['call_id'].is_unique and val_df['call_id'].is_unique

캐시 로드
29200 3640
{'고열': np.int64(5186), '구토': np.int64(4463), '두통': np.int64(2905), '복통': np.int64(6771), '어지러움': np.int64(6334), '열상': np.int64(3465), '오심': np.int64(3341), '전신쇠약': np.int64(5310), '호흡곤란': np.int64(3927)}


## 2. 토큰 길이 확인

토큰 수 분포를 보고 `MAX_LEN`을 정합니다. 512 초과는 2.7%뿐이라 최종 모델은 512를 사용했습니다. (256과 비교한 결과는 6절)

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


lens = np.array([len(tokenizer.encode(t)) for t in train_df['text']])
print(f"중앙값 {np.median(lens):.0f} | p75 {np.percentile(lens,75):.0f} | "
      f"p90 {np.percentile(lens,90):.0f} | p95 {np.percentile(lens,95):.0f} | 최대 {lens.max()}")
print(f"512 초과 비율: {(lens > 512).mean()*100:.1f}%")

config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/752k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (590 > 512). Running this sequence through the model will result in indexing errors


중앙값 230 | p75 306 | p90 397 | p95 460 | 최대 873
512 초과 비율: 2.7%


## 3. Dataset / DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader
from functools import partial

MAX_LEN = 512      # 최종 모델 설정 (256 비교 실험은 6절)
BATCH_SIZE = 16

def tokenize_df(df):
    enc = tokenizer(df['text'].tolist(), truncation=True, max_length=MAX_LEN,
                    return_token_type_ids=False)
    return enc['input_ids'], enc['attention_mask']

class SymptomDataset(Dataset):
    def __init__(self, df):
        self.call_ids = df['call_id'].tolist()
        self.input_ids, self.attn = tokenize_df(df)
        self.labels = df['labels'].tolist()
    def __len__(self):
        return len(self.call_ids)
    def __getitem__(self, i):
        return {'input_ids': self.input_ids[i], 'attention_mask': self.attn[i],
                'labels': self.labels[i], 'call_id': self.call_ids[i]}

def collate_fn(batch, tokenizer):
    """배치 안에서 가장 긴 길이에 맞춰 padding"""
    padded = tokenizer.pad(
        [{'input_ids': b['input_ids'], 'attention_mask': b['attention_mask']} for b in batch],
        padding='longest', return_tensors='pt')
    padded['labels'] = torch.tensor([b['labels'] for b in batch], dtype=torch.float)
    padded['call_id'] = [b['call_id'] for b in batch]
    return padded

train_ds, val_ds = SymptomDataset(train_df), SymptomDataset(val_df)
collate = partial(collate_fn, tokenizer=tokenizer)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, collate_fn=collate)
print(len(train_ds), len(val_ds))   # 29200 3640

## 4. 모델 정의

backbone 출력을 mean pooling(패딩 제외 평균)한 뒤 2층 분류기로 9개 로짓을 출력합니다. `attn` pooling은 비교 실험용입니다.

In [ ]:
import torch.nn as nn
from transformers import AutoModel

device = torch.device('cuda')

class SymptomClassifier(nn.Module):
    """KLUE-RoBERTa backbone + pooling + 2층 MLP 분류기"""
    def __init__(self, model_name=MODEL_NAME, num_classes=9, dropout=0.3, pooling='mean'):
        super().__init__()
        self.pooling = pooling
        self.backbone = AutoModel.from_pretrained(model_name, add_pooling_layer=False)
        h = self.backbone.config.hidden_size
        if pooling == 'attn':
            self.attn = nn.Linear(h, 1)
        self.classifier = nn.Sequential(
            nn.Linear(h, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, num_classes))

    def mean_pooling(self, x, mask):
        m = mask.unsqueeze(-1).float()
        return (x * m).sum(1) / m.sum(1).clamp(min=1e-9)

    def attn_pooling(self, x, mask):
        scores = self.attn(x).squeeze(-1).float()
        scores = scores.masked_fill(mask == 0, -1e4)
        w = torch.softmax(scores, dim=-1).unsqueeze(-1)
        return (x.float() * w).sum(1)

    def forward(self, input_ids, attention_mask):
        x = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        pooled = self.attn_pooling(x, attention_mask) if self.pooling == 'attn' \
                 else self.mean_pooling(x, attention_mask)
        return self.classifier(pooled)

## 5. 학습

`SEED`만 42 / 123 / 2026으로 바꿔 3번 실행했습니다. epoch마다 Validation Macro F1(threshold 0.5)로 best checkpoint를 저장하고, 3 epoch 연속 개선이 없으면 멈춥니다. Colab 끊김에 대비해 매 epoch `latest.pt`를 저장하고 이어서 학습할 수 있습니다.

In [ ]:
import time
from sklearn.metrics import f1_score, precision_score, recall_score
from transformers import get_linear_schedule_with_warmup
from torch.amp import autocast, GradScaler

# ===== 실험 설정 (최종: mean pooling + posw, seed 42/123/2026) =====
LOSS_TYPE = 'posw'   # 'posw' | 'sqrt' | 'none'
POOLING = 'mean'     # 'mean' | 'attn'
SEED = 42            # 42, 123, 2026으로 바꿔 3회 실행
# ==================================================================

EPOCHS = 8
PATIENCE = 3
RUN_NAME = f'klue-roberta_{POOLING}pool_{LOSS_TYPE}_seed{SEED}'   # 예: klue-roberta_meanpool_posw_seed42
RUN_DIR = f'{WORK_DIR}/{RUN_NAME}'
os.makedirs(RUN_DIR, exist_ok=True)
LATEST_CKPT = f'{RUN_DIR}/latest.pt'
BEST_CKPT   = f'{RUN_DIR}/best.pt'
LOG_PATH    = f'{RUN_DIR}/log.csv'

# pos_weight = Train의 (음성 수 / 양성 수). Validation 성능이 아니라 Train 통계로만 정함
y = np.array(train_df['labels'].tolist())
ratio = (len(y) - y.sum(0)) / y.sum(0)
if LOSS_TYPE == 'posw':
    pw = torch.tensor(ratio, dtype=torch.float).to(device)
elif LOSS_TYPE == 'sqrt':
    pw = torch.tensor(np.sqrt(ratio), dtype=torch.float).to(device)
else:
    pw = None
criterion = nn.BCEWithLogitsLoss(pos_weight=pw)

print(f"RUN: {RUN_NAME}")
print("pos_weight:", None if pw is None else pw.cpu().numpy().round(2))
print("재개 가능:", os.path.exists(LATEST_CKPT))

In [ ]:
def save_ckpt(path, epoch, best_f1, bad_epochs):
    """학습 재개용 체크포인트 (모델·optimizer·scheduler·scaler 상태)"""
    torch.save({'epoch': epoch, 'best_f1': best_f1, 'bad_epochs': bad_epochs,
                'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict()}, path)

def train_one_epoch():
    """1 epoch 학습 (AMP 사용), 평균 loss 반환"""
    model.train()
    total = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        with autocast('cuda'):
            logits = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            loss = criterion(logits.float(), batch['labels'].to(device))
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total += loss.item()
    return total / len(train_loader)

@torch.no_grad()
def evaluate(loader):
    """확률 → threshold 0.5로 판정 → Macro F1, 클래스별 F1 계산"""
    model.eval()
    probs, labels, ids = [], [], []
    for batch in loader:
        with autocast('cuda'):
            logits = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
        probs.append(torch.sigmoid(logits.float()).cpu())
        labels.append(batch['labels'])
        ids += batch['call_id']
    probs, labels = torch.cat(probs).numpy(), torch.cat(labels).numpy()
    preds = (probs >= THRESHOLD).astype(int)
    macro = f1_score(labels, preds, average='macro', zero_division=0)
    per_cls = f1_score(labels, preds, average=None, zero_division=0)
    return macro, per_cls, probs, labels, ids

def save_probs_csv(path, ids, probs, labels):
    """Validation 확률·정답 저장 (앙상블 조합 비교에 사용)"""
    df = pd.DataFrame({'call_id': ids})
    for i, s in enumerate(TARGET_SYMPTOMS):
        df[f'prob_{s}'] = probs[:, i]
    for i, s in enumerate(TARGET_SYMPTOMS):
        df[f'true_{s}'] = labels[:, i].astype(int)
    df.to_csv(path, index=False, encoding='utf-8-sig')

def pr_table(run_name):
    df = pd.read_csv(f'{WORK_DIR}/{run_name}/val_probs.csv')
    p = df[[f'prob_{s}' for s in TARGET_SYMPTOMS]].values >= THRESHOLD
    t = df[[f'true_{s}' for s in TARGET_SYMPTOMS]].values
    return pd.DataFrame({
        'P': precision_score(t, p, average=None, zero_division=0),
        'R': recall_score(t, p, average=None, zero_division=0),
        'F1': f1_score(t, p, average=None, zero_division=0)}, index=TARGET_SYMPTOMS).round(3)

In [ ]:
set_seed(SEED)
g = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate, generator=g)

model = SymptomClassifier(pooling=POOLING).to(device)
# backbone은 작은 lr, 새로 붙인 head는 큰 lr
backbone_ids = {id(p) for p in model.backbone.parameters()}
head_params = [p for p in model.parameters() if id(p) not in backbone_ids]   # classifier + attn
optimizer = torch.optim.AdamW([
    {'params': list(model.backbone.parameters()), 'lr': 2e-5},
    {'params': head_params, 'lr': 1e-3}], weight_decay=0.01)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
scaler = GradScaler('cuda')

start_epoch, best_f1, bad_epochs = 0, 0.0, 0
if os.path.exists(LATEST_CKPT):
    ck = torch.load(LATEST_CKPT, map_location=device)
    model.load_state_dict(ck['model']); optimizer.load_state_dict(ck['optimizer'])
    scheduler.load_state_dict(ck['scheduler']); scaler.load_state_dict(ck['scaler'])
    start_epoch, best_f1, bad_epochs = ck['epoch'] + 1, ck['best_f1'], ck['bad_epochs']
    print(f"재개: epoch {start_epoch+1}부터, best {best_f1:.4f}")

for epoch in range(start_epoch, EPOCHS):
    t0 = time.time()
    loss = train_one_epoch()
    macro, per_cls, probs, labels, ids = evaluate(val_loader)
    lr = scheduler.get_last_lr()[0]
    print(f"Epoch {epoch+1}/{EPOCHS} | loss {loss:.4f} | macro-F1 {macro:.4f} | "
          f"lr {lr:.2e} | {time.time()-t0:.0f}s")

    row = {'epoch': epoch+1, 'loss': loss, 'macro_f1': macro, 'lr': lr,
           **{f'f1_{s}': f for s, f in zip(TARGET_SYMPTOMS, per_cls)}}
    pd.DataFrame([row]).to_csv(LOG_PATH, mode='a', index=False,
                               header=not os.path.exists(LOG_PATH))

    if macro > best_f1:
        best_f1, bad_epochs = macro, 0
        save_ckpt(BEST_CKPT, epoch, best_f1, bad_epochs)
        save_probs_csv(f'{RUN_DIR}/val_probs.csv', ids, probs, labels)
        print(f"  → best 갱신 {best_f1:.4f}")
    else:
        bad_epochs += 1
    save_ckpt(LATEST_CKPT, epoch, best_f1, bad_epochs)
    if bad_epochs >= PATIENCE:
        print("Early stopping"); break

print(f"완료 | {RUN_NAME} | best macro-F1 {best_f1:.4f}")

del model, optimizer, scheduler, scaler
import gc; gc.collect(); torch.cuda.empty_cache()

### 학습 로그 (최종 3 seed)

위 학습 셀이 epoch마다 저장한 `log.csv`를 seed별로 출력합니다.

In [ ]:
# 최종 모델 3개의 epoch별 학습 로그
for seed in [42, 123, 2026]:
    run = f'klue-roberta_meanpool_posw_seed{seed}'
    log = pd.read_csv(f'{WORK_DIR}/{run}/log.csv')
    print(f"===== {run} | best macro-F1 {log['macro_f1'].max():.4f} (epoch {int(log.loc[log['macro_f1'].idxmax(), 'epoch'])}) =====")
    print(log[['epoch', 'loss', 'macro_f1', 'lr']].round(4).to_string(index=False), "\n")

## 6. 설정 비교: MAX_LEN 512 vs 256 (seed 42)

Macro F1 0.618 vs 0.614로 차이가 작아 512를 유지했습니다.

In [ ]:
# 클래스별 Precision / Recall / F1 비교
pcols = [f'prob_{c}' for c in TARGET_SYMPTOMS]
tcols = [f'true_{c}' for c in TARGET_SYMPTOMS]
from sklearn.metrics import precision_score, recall_score

def pr(run):
    df = pd.read_csv(f'{WORK_DIR}/{run}/val_probs.csv')
    p = df[pcols].values >= THRESHOLD
    tt = df[tcols].values
    return pd.DataFrame({
        'P': precision_score(tt, p, average=None, zero_division=0),
        'R': recall_score(tt, p, average=None, zero_division=0),
        'F1': f1_score(tt, p, average=None, zero_division=0)}, index=TARGET_SYMPTOMS)

cmp = pd.concat({'len512': pr('klue-roberta_meanpool_posw_seed42'),
                 'len256': pr('klue-roberta_meanpool_posw_len256_seed42')}, axis=1)
cmp.loc['macro'] = cmp.mean()
print(cmp.round(3))

      len512               len256              
           P      R     F1      P      R     F1
고열     0.582  0.726  0.646  0.625  0.721  0.669
구토     0.530  0.611  0.567  0.513  0.641  0.570
두통     0.387  0.612  0.474  0.357  0.615  0.451
복통     0.718  0.834  0.772  0.721  0.833  0.773
어지러움   0.605  0.699  0.649  0.596  0.687  0.638
열상     0.763  0.937  0.841  0.736  0.940  0.825
오심     0.288  0.566  0.382  0.287  0.573  0.383
전신쇠약   0.456  0.797  0.580  0.443  0.799  0.570
호흡곤란   0.589  0.720  0.648  0.578  0.724  0.643
macro  0.546  0.722  0.618  0.540  0.726  0.614


## 7. 앙상블 조합 선정

seed 3개의 확률을 평균한 뒤, 모델 조합별 Macro F1(threshold 0.5)을 비교합니다. Validation은 조합 선정에만 사용했습니다.

→ **KLUE posw + KcELECTRA none (0.6543)** 을 최종 제출 조합으로 선택

In [ ]:
# 모델 조합별 Validation Macro F1 비교 (seed 평균 → 모델 평균 → threshold 0.5)
import numpy as np, pandas as pd
from sklearn.metrics import f1_score

WORK_DIR = '/content/drive/MyDrive/DCC/mission3/klue_roberta'
S = ["고열","구토","두통","복통","어지러움","열상","오심","전신쇠약","호흡곤란"]
pc = [f'prob_{s}' for s in S]; tc = [f'true_{s}' for s in S]
SEEDS = [42, 123, 2026]
rd = lambda p: pd.read_csv(p).set_index('call_id').sort_index()

G = {
 'klue_posw': [rd(f'{WORK_DIR}/klue-roberta_meanpool_posw_seed{s}/val_probs.csv') for s in SEEDS],
 'klue_none': [rd(f'{WORK_DIR}/klue-roberta_meanpool_none_seed{s}/val_probs.csv') for s in SEEDS],
 'kc_none':   [rd(f'{WORK_DIR}/kcelectra/m3_kcelectra_base_s{s}_probs.csv') for s in SEEDS],
}
ref = G['klue_posw'][0]
for g in G.values():
    for d in g:
        assert (d.index == ref.index).all() and (d[tc].values == ref[tc].values).all()
t = ref[tc].values
mean = lambda g: np.mean([d[pc].values for d in g], axis=0)   # 그룹(모델) 안에서 seed 평균
M = {k: mean(v) for k, v in G.items()}

combos = {
 'KLUE posw':                M['klue_posw'],
 'Kc none':                  M['kc_none'],
 'KLUE posw + Kc none':      (M['klue_posw'] + M['kc_none']) / 2,
 'KLUE posw + KLUE none':    (M['klue_posw'] + M['klue_none']) / 2,
 '3그룹 (posw+none+Kc)':     (M['klue_posw'] + M['klue_none'] + M['kc_none']) / 3,
}
res = pd.DataFrame({k: f1_score(t, v >= 0.5, average=None, zero_division=0) for k, v in combos.items()}, index=S)
res.loc['macro'] = res.mean()
print(res.round(4))

a, b = M['klue_posw'] >= 0.5, M['kc_none'] >= 0.5
print(f"\nKLUE posw vs Kc none 예측 불일치율: {(a != b).mean():.3f}")
print(f"확률 상관계수: {np.corrcoef(M['klue_posw'].ravel(), M['kc_none'].ravel())[0,1]:.3f}")

       KLUE posw  Kc none  KLUE posw + Kc none  KLUE posw + KLUE none  \
고열        0.6685   0.6774               0.7031                 0.7025   
구토        0.5545   0.5915               0.6151                 0.6064   
두통        0.4423   0.5206               0.5492                 0.5449   
복통        0.8066   0.8211               0.8322                 0.8164   
어지러움      0.6555   0.6539               0.6564                 0.6605   
열상        0.8795   0.8820               0.8822                 0.8870   
오심        0.3785   0.1538               0.3913                 0.3863   
전신쇠약      0.5820   0.5532               0.5808                 0.5791   
호흡곤란      0.6542   0.6689               0.6783                 0.6708   
macro     0.6246   0.6136               0.6543                 0.6504   

       3그룹 (posw+none+Kc)  
고열                 0.6976  
구토                 0.6115  
두통                 0.5469  
복통                 0.8244  
어지러움               0.6617  
열상                 0.8867  
